# 一,本地部署推理

### 1.请求的预处理

In [28]:
from docutils.parsers.rst import roles
from jinja2.compiler import generate
from openpyxl.styles.builtins import output
from sympy.physics.units import temperature
from torch.xpu import device
from transformers import AutoTokenizer

# 初始化 Tokenizer
tokenizer = AutoTokenizer.from_pretrained("/Users/wjx/Documents/wjx/minimind/minimind-3")
# 示例输入
system_message = {"role": "system", "content": "你是一个历史老师，当学生询问你问题的时候，你总是中文进行回答。"}
user_input = "请问中国的四大发明是什么？"
message = [
    system_message,
    {"role": "user", "content": user_input}
]
# 分词+编码
input_ids = tokenizer.apply_chat_template(message, add_generation_prompt=True, return_tensors="pt")
# 转换消息为模型输入格式
print(f'输入：{user_input}\n编码：{input_ids}')

输入：请问中国的四大发明是什么？
编码：tensor([[   1,  118, 4849,  234,  441, 1001, 1874, 2152, 2025,  294,  899, 2278,
         6090,  441, 5523, 4524,  294,  441, 6056, 3606,  815, 3854,  302,    2,
          234,    1,  832,  311,  234, 5736, 1405,  296, 2405,  565,  532,  851,
         3013,  776,    2,  234,    1, 1388,  570,  811,  234,   25,  234,  234,
           26,  234,  234]])


### 2.推理过程

In [29]:
from transformers import AutoModelForCausalLM
import torch

# 加载模型
model_id = "/Users/wjx/Documents/wjx/minimind/minimind-3"
model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.bfloat16,  # 量化
    device_map="auto"
)
input_ids = input_ids.to(model.device)
terminators = [tokenizer.eos_token_id,
               tokenizer.convert_tokens_to_ids("<|eot_ids>")]
outputs = model.generate(input_ids,
                         max_new_tokens=256,
                         eos_token_id=terminators,
                         do_sample=True, temperature=0.6,
                         top_p=0.9)
print(f"推理结果：{outputs}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


推理结果：tensor([[   1,  118, 4849,  234,  441, 1001, 1874, 2152, 2025,  294,  899, 2278,
         6090,  441, 5523, 4524,  294,  441, 6056, 3606,  815, 3854,  302,    2,
          234,    1,  832,  311,  234, 5736, 1405,  296, 2405,  565,  532,  851,
         3013,  776,    2,  234,    1, 1388,  570,  811,  234,   25,  234,  234,
           26,  234,  234, 1405,  296, 2405,  565,  532,  851,  357, 1394, 3802,
          617,  345, 2357,  318,  151,  617,  345, 2329, 2381,  334, 1117, 2604,
         3142,  294, 1455,  369, 1460,  579,  296, 1209,  764, 6102,  423,  963,
         2252,  294,  461, 1950, 1874, 2636, 5686, 5086, 2214,  302,    2]],
       device='mps:0')


### 3.后处理与返回结果

In [30]:
import json
import xml.etree.ElementTree as ET

# 解码和过滤
output_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:93])
print(f"问题:{user_input}\n回复：{output_text}")

# json格式化
json_output = {"问题": user_input, "回复": output_text}
json_formatted = json.dumps(json_output, ensure_ascii=False, indent=4)
print("JSON格式化输出：")
print(json_formatted)

# xml格式化
root = ET.Element("Response")
question = ET.SubElement(root, "问题")
question.text = user_input
answer = ET.SubElement(root, "回复")
answer.text = output_text
xml_formatted = ET.tostring(root, encoding="utf-8").decode("utf-8")
print("XML格式化输出：")
print(xml_formatted)



问题:请问中国的四大发明是什么？
回复：中国的四大发明是造纸术、印刷术、火药和指南针，它们在各自的领域内发挥了重要作用，对世界历史的发展产生了深远的影响
JSON格式化输出：
{
    "问题": "请问中国的四大发明是什么？",
    "回复": "中国的四大发明是造纸术、印刷术、火药和指南针，它们在各自的领域内发挥了重要作用，对世界历史的发展产生了深远的影响"
}
XML格式化输出：
<Response><问题>请问中国的四大发明是什么？</问题><回复>中国的四大发明是造纸术、印刷术、火药和指南针，它们在各自的领域内发挥了重要作用，对世界历史的发展产生了深远的影响</回复></Response>


# 二，交互式推理

In [36]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "/Users/wjx/Documents/wjx/minimind/minimind-3"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)

system_message = {
    "role": "system",
    "content": "你是一个中文老师，必须用中文回答我的问题；现在你正在和学生讨论历史问题，这是个多轮对话场景，你需要结合历史谈话回答"
}

# 全局对话历史，后续对话一直往这里增加
message = [system_message]


def generate_response(user_input):
    # 用户输入加入历史
    message.append({"role": "user", "content": user_input})
    input_ids = tokenizer.apply_chat_template(message, add_generation_prompt=True, return_tensors="pt").to(model.device)

    # 终止标记
    terminators = [tokenizer.eos_token_id,
                   tokenizer.convert_tokens_to_ids("<|eot_id|>")]
    outputs = model.generate(input_ids,
                             max_new_tokens=256,
                             eos_token_id=terminators,
                             do_sample=True,
                             temperature=0.6,
                             top_p=0.9, )
    # 解码
    output_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    # 将模型回复增加到历史上下文中
    message.append({"role": "assistant", "content": output_text})
    return output_text


print("中文老师：同学！你可以问我任何问题，或者输入“退出”下课")

while True:
    user_input = input("You:")
    if user_input.lower() == "退出":
        print("下课")
        break
    else:
        reply = generate_response(user_input)
        print(f"回复：{reply}")

中文老师：同学！你可以问我任何问题，或者输入“退出”下课


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


回复：很高兴听到你最近在讨论历史问题。历史是一个复杂而多维的话题，涉及许多不同的方面。以下是一些可能的讨论点：

1. **古代文明与现代历史**：古代文明如古埃及、古希腊、古罗马等对世界历史有着深远的影响。这些文明在不同的时期和地区有着不同的发展和演变。例如，古埃及的金字塔和古罗马的金字塔在当时是世界上最高的建筑，而古希腊的宙斯则在公元前几世纪得到了广泛应用。

2. **科技与文明**：现代科技的发展对历史产生了深远的影响。例如，互联网的普及使得知识的传播和交流变得更加便捷。同时，工业革命也促进了知识的快速更新，如蒸汽机、电力、互联网等技术的快速发展。

3. **社会结构与政治制度**：历史事件往往对社会结构和政治制度产生深远影响。例如，二战后的世界格局发生了变化，引发了国际关系的复杂性。同时，历史事件的改变也影响了国家的政治制度，如中�


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


回复：刘备是一位中国历史上的著名政治家和军事家，他在政治、军事、文化和政治方面有着卓越的贡献。刘备在政治、军事、文化和政治方面都有着深刻的见解和贡献。

他曾经在公元前1世纪的三十年间担任重要职务，但最终在公元前1世纪的十年间，他因政治和军事权力而被视为“中国第一帝国”。刘备的军事才能和政治智慧在当时被视为“中国第一帝国”，他在历史上被认为是“中国第一帝国”。

刘备在政治上也有着卓越的贡献。他曾经在公元前1世纪的三十年间担任过重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


回复：诸葛亮是中国历史上的著名政治家和军事家，他在政治、军事、文化和政治方面都有着深远的影响。诸葛亮在政治、军事、文化和政治方面都有着卓越的贡献。

他曾经担任过重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任过重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公元前1世纪的十年间担任了重要职务，如公元前1世纪的十年间，他在公


KeyboardInterrupt: Interrupted by user